# Day 32 — Statistical inference: hypothesis testing, CIs
Objectives:
- One-sample and two-sample tests (t-test, chi-square).
- Confidence intervals.
- Practical interpretation and common pitfalls.

In [ ]:
import numpy as np, pandas as pd
from scipy import stats
rng = np.random.default_rng(0)
# One-sample t-test: is mean different from 0?
x = rng.normal(loc=0.2, scale=1.0, size=200)
t, p = stats.ttest_1samp(x, popmean=0.0)
t, p
# 95% CI for the mean
mean = x.mean(); se = x.std(ddof=1)/np.sqrt(len(x))
ci = (mean - 1.96*se, mean + 1.96*se); mean, ci


In [ ]:
# Two-sample t-test
y = rng.normal(loc=0.0, scale=1.0, size=220)
t2, p2 = stats.ttest_ind(x, y, equal_var=False)
t2, p2
# Chi-square test for independence (contingency table)
table = np.array([[30, 20],[15,35]])
chi2, pval, dof, exp = stats.chi2_contingency(table)
chi2, pval, dof


## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — sampling uncertainty, confidence intervals, and hypothesis-test evidence

### Mental model

Statistical inference reasons from a finite sample toward an unknown
population quantity. An estimator such as a sample mean varies across
hypothetical repeated samples; its **standard error** describes that
sampling variability. A confidence interval is a procedure with a
long-run coverage property, not a probability statement about a fixed
parameter after the data are observed.

A hypothesis test asks how incompatible an observed statistic is with
a precisely stated null model. The p-value is conditional on that null
model and its assumptions. It is not the probability that the null is
true, and it does not measure whether an effect is useful.

### Read the API before running it

- **`stats.ttest_1samp(sample, popmean)`:** compares a sample mean with a reference value using a t statistic and estimated standard error.
- **`stats.ttest_ind(a, b, equal_var=False)`:** runs Welch's two-sample test without assuming equal population variances.
- **`estimate ± critical_value * standard_error`:** forms an interval only after choosing a confidence level and an appropriate sampling model.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — see interval width respond to sample size

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** The observations are independent and the mean's t-based sampling model is reasonable.

In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(3201)

def mean_interval(sample, confidence=0.95):
    sample = np.asarray(sample, dtype=float)
    se = sample.std(ddof=1) / np.sqrt(sample.size)
    critical = stats.t.ppf((1 + confidence) / 2, df=sample.size - 1)
    return sample.mean() + np.array([-1, 1]) * critical * se

small = rng.normal(loc=2.0, scale=1.0, size=25)
large = rng.normal(loc=2.0, scale=1.0, size=400)
small_ci, large_ci = mean_interval(small), mean_interval(large)
print({"small": small_ci, "large": large_ci})
assert np.ptp(large_ci) < np.ptp(small_ci)

**Expected observation:** The larger sample normally produces a much narrower interval because its standard error is smaller.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — separate statistical detection from practical size

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** The operational decision depends on effect size and uncertainty, not a significance threshold alone.

In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(3202)
sample = rng.normal(loc=0.02, scale=1.0, size=100_000)
result = stats.ttest_1samp(sample, popmean=0.0)
standardized_effect = sample.mean() / sample.std(ddof=1)
print({"p_value": result.pvalue, "standardized_effect": standardized_effect})
assert abs(standardized_effect) < 0.05

**Expected observation:** A very small effect can have a small p-value when the sample is large.

### Debugging and practice ramp

**Common mistake:** Reporting `p < 0.05` without the estimate, interval, assumptions, sample size, or practical decision threshold.

**Diagnostic:** Reconstruct the test statistic from estimate divided by standard error, inspect the data-generating grain, and calculate an effect size.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define sampling uncertainty, confidence intervals, and hypothesis-test evidence in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Do not make a population claim when independence, selection, multiplicity, or the measurement process is unexplained.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## Learner exercises and progressive hints

1. Draw groups from `Normal(0, 1)` and `Normal(0.3, 1)`, then test the
   difference in means with Welch's t-test.

**Verify:** Practice 1 — sampling uncertainty, confidence intervals, and hypothesis-test evidence — record seed, both sample sizes, sample means/variances, Welch t statistic, degrees of freedom, and p-value; independently recompute the mean difference and state the exact null plus the decision at a declared alpha.

2. Build a contingency table from categorical data and run a chi-square test.
   For a fully offline run, construct a small table directly; a cached Seaborn
   dataset is optional.

**Verify:** Practice 2 — sampling uncertainty, confidence intervals, and hypothesis-test evidence — print the observed and expected contingency tables, chi-square statistic, degrees of freedom, and p-value; assert observed and expected totals match and flag any expected cell below 5.

3. Compute 90% and 99% confidence intervals for the same mean and compare their
   widths.

**Verify:** Practice 3 — sampling uncertainty, confidence intervals, and hypothesis-test evidence — from one unchanged sample, print its mean, standard error, and both interval endpoints; assert the 99% interval is wider than the 90% interval and both are centered on the same sample mean.

### Progressive hints

1. Use the same seeded generator, retain both sample sizes, and set
   `equal_var=False`. Interpret the estimated difference as well as the p-value.
2. Rows and columns represent category levels; cells contain counts, not raw
   labels. Inspect the expected counts returned by `chi2_contingency`.
3. Only the critical value changes when the sample and standard error remain
   fixed. Higher confidence should require a wider interval.

### Additional mastery practice

Separate effect estimation from decision thresholds. Report uncertainty, assumptions, and practical magnitude rather than treating a p-value as a verdict.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

4. **Prediction:** Hold the true mean difference and variance fixed, then predict how increasing each group's sample size from 20 to 200 affects standard error, confidence-interval width, power, and effect size.
   **Progressive hint:** Standard error shrinks approximately with 1/sqrt(n); the underlying standardized effect does not grow merely because more rows were collected.

**Verify:** Prediction — with a seeded simulation at n=20 and n=200 per group, print standard error, interval width, power, and standardized effect; verify standard error/width shrink by about sqrt(10), power rises, and the population effect size remains fixed.

5. **Implementation:** Build a seeded percentile-bootstrap confidence interval for a median difference. Validate empty groups and expose the number of resamples as a parameter.
   **Progressive hint:** Resample each group independently with replacement, compute one median difference per resample, then take symmetric quantiles.

**Verify:** Implementation — with a declared seed and at least 5,000 resamples, print observed median difference and percentile endpoints; assert repeatability, resample count, and a ValueError for either empty group.

6. **Multiple-comparison reasoning:** You test 20 unrelated null hypotheses at alpha=0.05. Estimate the chance of at least one false positive, then compare Bonferroni and false-discovery-rate control for a planned analysis.
   **Progressive hint:** Under independent true nulls, use 1-(1-alpha)**20. Bonferroni controls family-wise error; Benjamini-Hochberg targets the expected false-discovery proportion among rejections.

**Verify:** Multiple-comparison reasoning — show family-wise false-positive probability 1 - 0.95^20 (about 0.6415), Bonferroni per-test alpha 0.0025, and a sorted Benjamini-Hochberg decision table with original hypothesis order restored.

Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 4 — Prediction


# Practice 5 — Implementation


# Practice 6 — Multiple-comparison reasoning
